# AllSci Application Crawler & Field Mapper

This notebook performs comprehensive crawling of the entire AllSci application to:
1. **Discover all pages** (list pages, detail pages, tabs)
2. **Build application sitemap** (structure and relationships)
3. **Map all data fields** on every page and tab
4. **Generate comprehensive documentation** of the application structure

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urlparse, urljoin, parse_qs
import json
from datetime import datetime
from collections import defaultdict, deque
import hashlib

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Login credentials
SUPABASE_EMAIL = "rlalani@allsci.com"
SUPABASE_PASSWORD = "!!Casio1994$$"

# Base configuration
BASE_URL = "https://app.allsci.com"
LOGIN_URL = "https://app.allsci.com/?login=true"

# Search queries - starting points for crawling
# The application uses search-based navigation, not static list pages
SEARCH_QUERIES = [
    "covid",        # Example: finds hypotheses, articles, trials, grants, researchers
    "cancer",       # Different search term to discover different data
    # Add more search terms to discover different parts of the application
]

# Search result tabs to explore (these appear on search results)
SEARCH_RESULT_TABS = [
    'Hypotheses',
    'Articles',
    'Clinical Trials',
    'Grants',
    'Researchers',
]

# Additional navigation URLs (if any exist)
ADDITIONAL_URLS = [
    "https://app.allsci.com/explore/clinical-trials",  # Atlas view
    # Add other known URLs here
]

# Crawling limits
MAX_PAGES_TO_CRAWL = 50  # Limit total pages (increase for full crawl)
MAX_RESULTS_PER_SEARCH_TAB = 5  # How many detail pages to visit from each search result tab
MAX_DEPTH = 3  # How many levels deep to crawl

# Wait times
PAGE_LOAD_WAIT = 15  # seconds
ELEMENT_WAIT = 10    # seconds
TAB_SWITCH_WAIT = 3  # seconds
SEARCH_WAIT = 5      # seconds to wait for search results

# URL patterns to identify page types
PAGE_TYPE_PATTERNS = {
    'search_results': r'/search\?query=',
    'clinical_trial_detail': r'/clinical-trial/ASC-CT-\d+',
    'explore_atlas': r'/explore/clinical-trials',
    'work_detail': r'/work/ASC-WK-\d+',
    'patent_detail': r'/patent/ASC-PT-\d+',
    'hypothesis_detail': r'/hypothesis/ASC-HY-\d+',
    'grant_detail': r'/grant/ASC-GR-\d+',
    'researcher_detail': r'/researcher/ASC-RS-\d+',
}

# Detail page tab patterns (tabs that appear on detail pages like trials, works, etc.)
DETAIL_PAGE_TAB_PATTERNS = [
    'Overview',
    'Works',
    'Hypotheses',
    'Patents',
    'Related Trials',
    'Timeline',
    'Organizations',
    'People',
    'Funding',
]

## Helper Functions

In [ ]:
def setup_driver(headless=False):
    """Setup Chrome WebDriver."""
    chrome_options = Options()
    if headless:
        chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver


def login_to_application(driver, login_url, email, password):
    """Login to the application."""
    print(f"Logging in to: {login_url}")
    driver.get(login_url)
    
    try:
        email_input = WebDriverWait(driver, ELEMENT_WAIT).until(
            EC.presence_of_element_located((By.NAME, "email"))
        )
        email_input.send_keys(email)
        
        password_input = driver.find_element(By.NAME, "password")
        password_input.send_keys(password)
        
        sign_in_button = driver.find_element(By.XPATH, "//button[@type='submit' and contains(text(), 'Sign In')]")
        sign_in_button.click()
        
        time.sleep(5)
        print("✓ Login successful!")
        return True
        
    except Exception as e:
        print(f"✗ Login failed: {e}")
        return False


def wait_for_page_load(driver, timeout=PAGE_LOAD_WAIT):
    """Wait for page to finish loading."""
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: d.execute_script('return document.readyState') == 'complete'
        )
        time.sleep(2)
    except TimeoutException:
        pass


def normalize_url(url, base_url=BASE_URL):
    """Normalize URL for comparison."""
    # Remove fragments
    url = url.split('#')[0]
    # Remove trailing slashes for consistency
    url = url.rstrip('/')
    # Make absolute
    if not url.startswith('http'):
        url = urljoin(base_url, url)
    return url


def get_page_type(url):
    """Identify page type based on URL pattern."""
    for page_type, pattern in PAGE_TYPE_PATTERNS.items():
        if re.search(pattern, url):
            return page_type
    return 'unknown'


def get_url_hash(url):
    """Generate a hash for URL deduplication."""
    return hashlib.md5(normalize_url(url).encode()).hexdigest()[:12]

## Link Discovery Functions

In [ ]:
def discover_links(driver, base_url=BASE_URL):
    """Discover all internal links on the current page."""
    links = set()
    
    try:
        # Get all anchor tags
        anchor_elements = driver.find_elements(By.TAG_NAME, "a")
        
        for anchor in anchor_elements:
            try:
                href = anchor.get_attribute('href')
                if href:
                    # Normalize and check if internal
                    normalized = normalize_url(href, base_url)
                    if normalized.startswith(base_url):
                        links.add(normalized)
            except StaleElementReferenceException:
                continue
    except Exception as e:
        print(f"    Warning: Error discovering links: {e}")
    
    return links


def discover_tabs(driver):
    """Discover tabs/sections on detail pages (Overview, Works, Hypotheses, etc.)."""
    tabs = []
    
    # Common tab selectors
    tab_selectors = [
        "button[role='tab']",
        "a[role='tab']",
        "div[role='tab']",
        ".MuiTab-root",
        "[class*='tab']",
    ]
    
    for selector in tab_selectors:
        try:
            tab_elements = driver.find_elements(By.CSS_SELECTOR, selector)
            
            for tab in tab_elements:
                try:
                    tab_text = tab.text.strip()
                    if tab_text and len(tab_text) < 50:  # Reasonable tab name length
                        # Check if it's a known detail page tab pattern
                        if any(pattern.lower() in tab_text.lower() for pattern in DETAIL_PAGE_TAB_PATTERNS):
                            tabs.append({
                                'name': tab_text,
                                'element': tab,
                                'selector': selector
                            })
                except:
                    continue
        except:
            continue
    
    # Deduplicate by name
    seen_names = set()
    unique_tabs = []
    for tab in tabs:
        if tab['name'] not in seen_names:
            seen_names.add(tab['name'])
            unique_tabs.append(tab)
    
    return unique_tabs

In [ ]:
def perform_search(driver, query, base_url=BASE_URL):
    """Perform a search query and return the search results URL."""
    search_url = f"{base_url}/search?query={query}"
    print(f"  Performing search: '{query}'")
    driver.get(search_url)
    wait_for_page_load(driver)
    time.sleep(SEARCH_WAIT)
    return search_url


def discover_search_result_tabs(driver):
    """Discover search result tabs (Hypotheses, Articles, Clinical Trials, etc.)."""
    tabs = []
    
    try:
        # Look for tab elements in search results
        # Pattern: div with "flex flex-row cursor-pointer" containing tab name
        tab_elements = driver.find_elements(By.CSS_SELECTOR, "div.flex.flex-row.cursor-pointer")
        
        for tab_elem in tab_elements:
            try:
                # Get the tab name
                tab_name_elem = tab_elem.find_element(By.CSS_SELECTOR, "p.MuiTypography-root")
                tab_name = tab_name_elem.text.strip()
                
                # Get the count if available
                count_elem = tab_elem.find_element(By.CSS_SELECTOR, "p.MuiTypography-root + p")
                count = count_elem.text.strip() if count_elem else "0"
                
                if tab_name and tab_name in SEARCH_RESULT_TABS:
                    tabs.append({
                        'name': tab_name,
                        'count': count,
                        'element': tab_elem
                    })
            except:
                continue
    except Exception as e:
        print(f"    Warning: Could not discover search tabs: {e}")
    
    return tabs


def extract_filter_fields(driver):
    """Extract filter fields from the search results sidebar."""
    fields = []
    
    try:
        # Find accordion/filter sections
        accordions = driver.find_elements(By.CSS_SELECTOR, ".MuiAccordion-root")
        
        for accordion in accordions:
            try:
                # Get filter name
                filter_name_elem = accordion.find_element(By.CSS_SELECTOR, ".MuiAccordionSummary-content h1")
                filter_name = filter_name_elem.text.strip()
                
                # Get filter options
                options = accordion.find_elements(By.CSS_SELECTOR, "label")
                
                for option in options[:5]:  # Sample first 5 options
                    try:
                        option_text = option.text.strip()
                        if option_text:
                            # Split label and count if present
                            match = re.match(r'^(.+?)\s*\(([0-9,]+)\)$', option_text)
                            if match:
                                label = match.group(1)
                                count = match.group(2)
                            else:
                                label = option_text
                                count = ""
                            
                            fields.append({
                                'category': 'search_filter',
                                'label': f"{filter_name} - {label}",
                                'value': count,
                                'selector': get_css_selector(option, driver),
                                'element_type': 'filter',
                                'has_data': bool(count)
                            })
                    except:
                        continue
            except:
                continue
    except Exception as e:
        print(f"    Warning: Could not extract filters: {e}")
    
    return fields


def extract_search_result_metrics(driver):
    """Extract result counts from search tabs."""
    fields = []
    
    try:
        # Find tab elements with counts
        tab_divs = driver.find_elements(By.CSS_SELECTOR, "div.flex.flex-row.cursor-pointer")
        
        for tab_div in tab_divs:
            try:
                # Get tab name
                name_elem = tab_div.find_element(By.CSS_SELECTOR, "p.MuiTypography-root:first-child")
                tab_name = name_elem.text.strip()
                
                # Get count
                count_elem = tab_div.find_element(By.CSS_SELECTOR, "p.MuiTypography-root + p")
                count = count_elem.text.strip()
                
                if tab_name and count:
                    fields.append({
                        'category': 'search_result_count',
                        'label': tab_name,
                        'value': count,
                        'selector': get_css_selector(tab_div, driver),
                        'element_type': 'tab_metric',
                        'has_data': True
                    })
            except:
                continue
    except Exception as e:
        pass
    
    return fields


def crawl_search_results(driver, query, depth=0):
    """Crawl search results for a given query, including all tabs."""
    all_pages_data = []
    discovered_links = []
    
    # Perform search
    search_url = perform_search(driver, query)
    
    # Create page data for search results page
    search_page_data = {
        'url': search_url,
        'page_type': 'search_results',
        'title': driver.title,
        'depth': depth,
        'tabs': [],
        'fields': [],
        'links': set(),
        'crawl_timestamp': datetime.now().isoformat(),
        'search_query': query
    }
    
    # Extract search result metrics (tab counts)
    print(f"    Extracting search result metrics...")
    search_page_data['fields'].extend(extract_search_result_metrics(driver))
    
    # Extract filter fields
    print(f"    Extracting filter fields...")
    search_page_data['fields'].extend(extract_filter_fields(driver))
    
    # Add context to fields
    for field in search_page_data['fields']:
        field['page_url'] = search_url
        field['page_title'] = driver.title
        field['tab_name'] = None
        field['page_type'] = 'search_results'
    
    all_pages_data.append(search_page_data)
    
    # Discover and click through search result tabs
    tabs = discover_search_result_tabs(driver)
    if tabs:
        print(f"    Found {len(tabs)} search result tabs: {', '.join([t['name'] for t in tabs])}")
        
        for tab in tabs:
            try:
                print(f"      → Clicking search tab: {tab['name']} ({tab['count']} results)")
                tab['element'].click()
                time.sleep(TAB_SWITCH_WAIT)
                
                # Extract fields from this tab view
                tab_fields = extract_all_fields(driver, search_url, tab_name=tab['name'])
                search_page_data['fields'].extend(tab_fields)
                search_page_data['tabs'].append(tab['name'])
                
                # Discover links in this tab's results
                result_links = discover_links(driver)
                
                # Filter to only detail page links (not other search links)
                detail_links = [link for link in result_links 
                              if any(pattern in link for pattern in ['ASC-CT-', 'ASC-WK-', 'ASC-PT-', 'ASC-HY-', 'ASC-GR-', 'ASC-RS-'])]
                
                # Sample limited number of results
                sampled_links = detail_links[:MAX_RESULTS_PER_SEARCH_TAB]
                discovered_links.extend(sampled_links)
                
                print(f"        Found {len(detail_links)} detail pages, sampling {len(sampled_links)}")
                
            except Exception as e:
                print(f"        Warning: Could not process search tab {tab['name']}: {e}")
                continue
    
    print(f"    ✓ Search results page: {len(search_page_data['fields'])} fields, {len(discovered_links)} detail pages discovered")
    
    return all_pages_data, discovered_links

## Search-Based Navigation Functions

## Field Extraction Functions

In [ ]:
def crawl_page_with_tabs(driver, url, depth=0):
    """Crawl a single page including all its tabs."""
    page_data = {
        'url': url,
        'page_type': get_page_type(url),
        'title': '',
        'depth': depth,
        'tabs': [],
        'fields': [],
        'links': set(),
        'crawl_timestamp': datetime.now().isoformat()
    }
    
    try:
        print(f"  {'  ' * depth}Crawling: {url}")
        driver.get(url)
        wait_for_page_load(driver)
        
        page_data['title'] = driver.title
        
        # Extract fields from main page
        print(f"  {'  ' * depth}  - Extracting fields from main page")
        main_fields = extract_all_fields(driver, url)
        page_data['fields'].extend(main_fields)
        
        # Discover tabs (for detail pages)
        tabs = discover_tabs(driver)
        if tabs:
            print(f"  {'  ' * depth}  - Found {len(tabs)} tabs: {', '.join([t['name'] for t in tabs])}")
            
            for tab in tabs:
                try:
                    print(f"  {'  ' * depth}    → Clicking tab: {tab['name']}")
                    tab['element'].click()
                    time.sleep(TAB_SWITCH_WAIT)
                    
                    # Extract fields from this tab
                    tab_fields = extract_all_fields(driver, url, tab_name=tab['name'])
                    page_data['fields'].extend(tab_fields)
                    page_data['tabs'].append(tab['name'])
                    
                    print(f"  {'  ' * depth}      Extracted {len(tab_fields)} fields")
                except Exception as e:
                    print(f"  {'  ' * depth}      Warning: Could not process tab {tab['name']}: {e}")
                    continue
        
        # Discover links
        links = discover_links(driver)
        page_data['links'] = links
        print(f"  {'  ' * depth}  - Found {len(links)} links")
        
        print(f"  {'  ' * depth}✓ Total fields extracted: {len(page_data['fields'])}")
        
    except Exception as e:
        print(f"  {'  ' * depth}✗ Error crawling {url}: {e}")
    
    return page_data


def crawl_application(driver, search_queries, additional_urls=[], max_pages=MAX_PAGES_TO_CRAWL, max_depth=MAX_DEPTH):
    """Crawl the entire application starting from search queries and additional URLs."""
    
    visited = set()
    to_visit = deque()
    
    all_pages = []
    all_fields = []
    sitemap = defaultdict(list)  # parent_url -> [child_urls]
    
    page_count = 0
    
    print(f"\n{'='*60}")
    print(f"PHASE 1: SEARCH-BASED DISCOVERY")
    print(f"{'='*60}\n")
    
    # First, crawl all search queries
    for query in search_queries:
        if page_count >= max_pages:
            break
        
        print(f"\n[Search Query: '{query}']")
        
        try:
            # Crawl search results for this query
            search_pages, discovered_links = crawl_search_results(driver, query, depth=0)
            
            # Add search pages to results
            all_pages.extend(search_pages)
            for page in search_pages:
                all_fields.extend(page['fields'])
                page_count += 1
            
            # Add discovered links to queue for detailed crawling
            for link in discovered_links:
                if link not in visited:
                    to_visit.append((link, 1))  # depth 1 since they're from search
                    sitemap[search_pages[0]['url']].append(link)
            
            print(f"  Added {len(discovered_links)} detail pages to queue")
            
        except Exception as e:
            print(f"  ✗ Error processing search query '{query}': {e}")
            continue
    
    # Add any additional URLs to the queue
    for url in additional_urls:
        if url not in visited:
            to_visit.append((url, 0))
    
    print(f"\n{'='*60}")
    print(f"PHASE 2: DETAIL PAGE CRAWLING")
    print(f"{'='*60}\n")
    print(f"Queue: {len(to_visit)} pages to crawl\n")
    
    # Now crawl discovered detail pages
    while to_visit and page_count < max_pages:
        url, depth = to_visit.popleft()
        
        # Skip if already visited or too deep
        if url in visited or depth > max_depth:
            continue
        
        visited.add(url)
        page_count += 1
        
        print(f"\n[{page_count}/{max_pages}] Depth {depth}")
        
        # Crawl the page
        page_data = crawl_page_with_tabs(driver, url, depth)
        all_pages.append(page_data)
        all_fields.extend(page_data['fields'])
        
        # Add discovered links to queue (but limit based on page type)
        page_type = page_data['page_type']
        
        # For detail pages, follow related links (but limit them)
        if 'detail' in page_type:
            detail_links = [link for link in page_data['links'] 
                          if any(pattern in link for pattern in ['ASC-CT-', 'ASC-WK-', 'ASC-PT-', 'ASC-HY-'])]
            links_to_follow = detail_links[:3]  # Max 3 related pages per detail page
        else:
            links_to_follow = []
        
        for link in links_to_follow:
            if link not in visited:
                to_visit.append((link, depth + 1))
                sitemap[url].append(link)
        
        # Show progress
        print(f"  Queue size: {len(to_visit)} | Visited: {len(visited)}")
    
    print(f"\n{'='*60}")
    print(f"CRAWL COMPLETE!")
    print(f"  Pages crawled: {page_count}")
    print(f"  Total fields extracted: {len(all_fields)}")
    print(f"{'='*60}")
    
    return {
        'pages': all_pages,
        'fields': all_fields,
        'sitemap': dict(sitemap),
        'visited_urls': list(visited)
    }

## Page Crawling Functions

In [ ]:
# Setup and login
print("Setting up crawler...\n")
driver = setup_driver(headless=False)

# Login
login_success = login_to_application(driver, LOGIN_URL, SUPABASE_EMAIL, SUPABASE_PASSWORD)

if not login_success:
    print("Login failed. Please check credentials.")
    driver.quit()
else:
    print("\nStarting application crawl...\n")
    print(f"{'='*60}")
    print(f"Configuration:")
    print(f"  Max pages: {MAX_PAGES_TO_CRAWL}")
    print(f"  Max depth: {MAX_DEPTH}")
    print(f"  Results per search tab: {MAX_RESULTS_PER_SEARCH_TAB}")
    print(f"  Search queries: {len(SEARCH_QUERIES)}")
    for query in SEARCH_QUERIES:
        print(f"    - '{query}'")
    if ADDITIONAL_URLS:
        print(f"  Additional URLs: {len(ADDITIONAL_URLS)}")
        for url in ADDITIONAL_URLS:
            print(f"    - {url}")
    print(f"{'='*60}\n")
    
    # Crawl
    crawl_results = crawl_application(driver, SEARCH_QUERIES, ADDITIONAL_URLS)
    
    # Cleanup
    driver.quit()

## Run the Crawler

In [ ]:
# Setup and login
print("Setting up crawler...\n")
driver = setup_driver(headless=False)

# Login
login_success = login_to_application(driver, LOGIN_URL, SUPABASE_EMAIL, SUPABASE_PASSWORD)

if not login_success:
    print("Login failed. Please check credentials.")
    driver.quit()
else:
    print("\nStarting application crawl...\n")
    print(f"{'='*60}")
    print(f"Configuration:")
    print(f"  Max pages: {MAX_PAGES_TO_CRAWL}")
    print(f"  Max depth: {MAX_DEPTH}")
    print(f"  Detail pages per list: {MAX_DETAIL_PAGES_PER_LIST}")
    print(f"  Seed URLs: {len(SEED_URLS)}")
    for url in SEED_URLS:
        print(f"    - {url}")
    print(f"{'='*60}\n")
    
    # Crawl
    crawl_results = crawl_application(driver, SEED_URLS)
    
    # Cleanup
    driver.quit()

## Generate Reports

In [ ]:
# Create DataFrames
df_fields = pd.DataFrame(crawl_results['fields'])
df_pages = pd.DataFrame([{
    'url': p['url'],
    'page_type': p['page_type'],
    'title': p['title'],
    'depth': p['depth'],
    'num_tabs': len(p['tabs']),
    'tabs': ', '.join(p['tabs']),
    'num_fields': len(p['fields']),
    'num_links': len(p['links']),
} for p in crawl_results['pages']])

print("\n" + "="*60)
print("CRAWL SUMMARY")
print("="*60)
print(f"\nTotal Pages: {len(df_pages)}")
print(f"Total Fields: {len(df_fields)}")
print(f"\nPages by Type:")
print(df_pages['page_type'].value_counts())
print(f"\nFields by Category:")
print(df_fields['category'].value_counts())
print(f"\nTop 10 Pages by Field Count:")
print(df_pages.nlargest(10, 'num_fields')[['url', 'page_type', 'num_fields', 'num_tabs']])

## Application Structure Map

In [ ]:
# Generate application structure
print("\n" + "="*60)
print("APPLICATION STRUCTURE MAP")
print("="*60)

for page in crawl_results['pages'][:20]:  # Show first 20
    indent = "  " * page['depth']
    print(f"\n{indent}📄 {page['url'].replace(BASE_URL, '')}")
    print(f"{indent}   Type: {page['page_type']} | Fields: {len(page['fields'])} | Links: {len(page['links'])}")
    if page['tabs']:
        print(f"{indent}   Tabs: {', '.join(page['tabs'])}")

## Field Coverage Analysis

In [ ]:
# Analyze field coverage
print("\n" + "="*60)
print("FIELD COVERAGE BY PAGE TYPE")
print("="*60)

coverage = df_fields.groupby(['page_type', 'category']).agg({
    'label': 'count',
    'has_data': 'sum'
}).rename(columns={'label': 'total_fields', 'has_data': 'fields_with_data'})

print(coverage)

# Show unique field labels by page type
print("\n" + "="*60)
print("UNIQUE FIELD LABELS BY PAGE TYPE")
print("="*60)

for page_type in df_fields['page_type'].unique():
    print(f"\n{page_type.upper()}:")
    type_fields = df_fields[df_fields['page_type'] == page_type]
    unique_labels = type_fields['label'].unique()[:15]
    for label in unique_labels:
        print(f"  - {label}")
    if len(type_fields['label'].unique()) > 15:
        print(f"  ... and {len(type_fields['label'].unique()) - 15} more")

## Tab Analysis

In [ ]:
# Analyze tabs
print("\n" + "="*60)
print("TAB ANALYSIS")
print("="*60)

tab_fields = df_fields[df_fields['tab_name'].notna()]
if len(tab_fields) > 0:
    print(f"\nTotal fields from tabs: {len(tab_fields)}")
    print(f"\nFields by Tab:")
    print(tab_fields.groupby('tab_name')['label'].count().sort_values(ascending=False))
    
    print(f"\nPages with Tabs:")
    pages_with_tabs = df_pages[df_pages['num_tabs'] > 0]
    print(pages_with_tabs[['url', 'page_type', 'tabs', 'num_tabs']].to_string())
else:
    print("\nNo tabs were detected on any pages.")

## Export Results

In [ ]:
# Export all data
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Field mapping CSV
field_file = f'field_mapping_full_{timestamp}.csv'
df_fields.to_csv(field_file, index=False)
print(f"\n✓ Field mapping exported to: {field_file}")

# 2. Page structure CSV
page_file = f'page_structure_{timestamp}.csv'
df_pages.to_csv(page_file, index=False)
print(f"✓ Page structure exported to: {page_file}")

# 3. Complete crawl data JSON
json_file = f'crawl_results_{timestamp}.json'
with open(json_file, 'w') as f:
    json.dump({
        'pages': [{
            **p,
            'links': list(p['links'])  # Convert sets to lists for JSON
        } for p in crawl_results['pages']],
        'sitemap': crawl_results['sitemap'],
        'visited_urls': crawl_results['visited_urls'],
        'summary': {
            'total_pages': len(crawl_results['pages']),
            'total_fields': len(crawl_results['fields']),
            'crawl_timestamp': timestamp
        }
    }, f, indent=2)
print(f"✓ Complete crawl data exported to: {json_file}")

# 4. Sitemap visualization
sitemap_file = f'sitemap_{timestamp}.txt'
with open(sitemap_file, 'w') as f:
    f.write("APPLICATION SITEMAP\n")
    f.write("="*80 + "\n\n")
    
    for page in crawl_results['pages']:
        indent = "  " * page['depth']
        f.write(f"{indent}{page['url']}\n")
        f.write(f"{indent}  Type: {page['page_type']}\n")
        f.write(f"{indent}  Fields: {len(page['fields'])}\n")
        if page['tabs']:
            f.write(f"{indent}  Tabs: {', '.join(page['tabs'])}\n")
        f.write("\n")

print(f"✓ Sitemap exported to: {sitemap_file}")

print(f"\n{'='*60}")
print("All exports complete!")
print(f"{'='*60}")